In [26]:
!pip install librosa tqdm scikit-learn tensorflow scipy

import os
import numpy as np
import pandas as pd
import librosa
import scipy.fftpack
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization


# ====================== PATH SETUP ======================
BASE_PATH = r"C:\Users\chaitu\Documents\sample_project_1\freesound-audio-tagging"

TRAIN_AUDIO_PATH = os.path.join(BASE_PATH, "audio_train")
TEST_AUDIO_PATH = os.path.join(BASE_PATH, "audio_test")
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")

# ====================== LOAD CSV ======================
df = pd.read_csv(TRAIN_CSV)

print(df.head())
print(df['label'].value_counts())

# ====================== SELECT SMALL SUBSET ======================
selected_classes = df['label'].value_counts().index[:5]
df = df[df['label'].isin(selected_classes)].copy()

MAX_SAMPLES_PER_CLASS = 100000
df = df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), MAX_SAMPLES_PER_CLASS), random_state=42)
).reset_index(drop=True)

print(df['label'].value_counts())
print("Total samples used:", len(df))

# ====================== AUDIO / MFCC SETTINGS ======================
SR = 16000
DURATION = 1.0
N_MFCC = 13
MAX_FRAMES = 32

# ====================== AUDIO FUNCTIONS ======================
def load_audio(file_path):
    y, _ = librosa.load(file_path, sr=SR)
    return y

def pad_audio(y):
    max_len = int(SR * DURATION)
    if len(y) < max_len:
        y = np.pad(y, (0, max_len - len(y)))
    else:
        y = y[:max_len]
    return y

def mfcc_from_scratch(signal, sr=16000, num_mfcc=13):
    if len(signal) == 0:
        return np.zeros((1, num_mfcc))

    # ====================== 1. PRE-EMPHASIS ======================
    pre_emphasis = 0.97
    emphasized_signal = np.append(signal[0], signal[1:] - pre_emphasis * signal[:-1])

    # ====================== 2. FRAMING ======================
    frame_size = 0.025
    frame_stride = 0.01

    frame_length = int(frame_size * sr)
    frame_step = int(frame_stride * sr)

    signal_length = len(emphasized_signal)
    num_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step)) + 1

    pad_signal_length = num_frames * frame_step + frame_length
    z = np.zeros((pad_signal_length - signal_length))
    pad_signal = np.append(emphasized_signal, z)

    indices = np.tile(np.arange(0, frame_length), (num_frames, 1)) + \
              np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T

    frames = pad_signal[indices.astype(np.int32, copy=False)]

    # ====================== 3. WINDOWING ======================
    frames *= np.hamming(frame_length)

    # ====================== 4. FFT & POWER SPECTRUM ======================
    NFFT = 512
    mag_frames = np.absolute(np.fft.rfft(frames, NFFT))
    pow_frames = ((1.0 / NFFT) * (mag_frames ** 2))

    # ====================== 5. MEL FILTER BANK ======================
    nfilt = 26

    def hz_to_mel(hz):
        return 2595 * np.log10(1 + hz / 700)

    def mel_to_hz(mel):
        return 700 * (10**(mel / 2595) - 1)

    low_mel = hz_to_mel(0)
    high_mel = hz_to_mel(sr / 2)

    mel_points = np.linspace(low_mel, high_mel, nfilt + 2)
    hz_points = mel_to_hz(mel_points)

    bins = np.floor((NFFT + 1) * hz_points / sr)

    fbank = np.zeros((nfilt, int(np.floor(NFFT / 2 + 1))))

    for m in range(1, nfilt + 1):
        f_m_minus = int(bins[m - 1])
        f_m = int(bins[m])
        f_m_plus = int(bins[m + 1])

        for k in range(f_m_minus, f_m):
            if bins[m] != bins[m - 1]:
                fbank[m - 1, k] = (k - bins[m - 1]) / (bins[m] - bins[m - 1])
        for k in range(f_m, f_m_plus):
            if bins[m + 1] != bins[m]:
                fbank[m - 1, k] = (bins[m + 1] - k) / (bins[m + 1] - bins[m])

    filter_banks = np.dot(pow_frames, fbank.T)
    filter_banks = np.where(filter_banks == 0, np.finfo(float).eps, filter_banks)

    # ====================== 6. LOG ======================
    log_fbank = np.log(filter_banks)

    # ====================== 7. DCT ======================
    mfcc = scipy.fftpack.dct(log_fbank, type=2, axis=1, norm='ortho')[:, :num_mfcc]

    # ====================== 8. MEAN NORMALIZATION ======================
    mfcc -= (np.mean(mfcc, axis=0) + 1e-8)

    return mfcc

def extract_features(y):
    mfcc = mfcc_from_scratch(y, sr=SR, num_mfcc=N_MFCC)

    if mfcc.shape[0] < MAX_FRAMES:
        pad_width = MAX_FRAMES - mfcc.shape[0]
        mfcc = np.pad(mfcc, ((0, pad_width), (0, 0)), mode='constant')
    else:
        mfcc = mfcc[:MAX_FRAMES, :]

    mfcc = mfcc.T
    mfcc = mfcc[..., np.newaxis]
    return mfcc

# ====================== PREPARE DATA ======================
X = []
y = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    file_path = os.path.join(TRAIN_AUDIO_PATH, row['fname'])
    try:
        audio = load_audio(file_path)
        audio = pad_audio(audio)
        features = extract_features(audio)
        X.append(features)
        y.append(row['label'])
    except Exception as e:
        print(f"Skipped {file_path}: {e}")

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

# ====================== ENCODE LABELS ======================
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# ====================== TRAIN-TEST SPLIT ======================
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# ====================== BUILD CNN MODEL ======================
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(N_MFCC, MAX_FRAMES, 1)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(len(le.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ====================== TRAIN MODEL ======================
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=16
)

# ====================== SAVE MODEL ======================
os.makedirs("models", exist_ok=True)
model.save("models/audio_cnn_mfcc.keras")

# ====================== PREDICTION FUNCTION ======================
def predict(file_path):
    audio = load_audio(file_path)
    audio = pad_audio(audio)
    features = extract_features(audio)
    features = np.expand_dims(features, axis=0)

    pred = model.predict(features, verbose=0)
    predicted_class = le.inverse_transform([np.argmax(pred)])[0]
    return predicted_class

# ====================== TEST PREDICTION ======================


from sklearn.metrics import f1_score

# Predictions on validation set
y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)

# F1 Score (macro = balanced across classes)
f1 = f1_score(y_val, y_pred, average='macro')

print("F1 Score:", f1)

print("F1 (macro):", f1_score(y_val, y_pred, average='macro'))
print("F1 (micro):", f1_score(y_val, y_pred, average='micro'))
print("F1 (weighted):", f1_score(y_val, y_pred, average='weighted'))

from sklearn.metrics import classification_report

print(classification_report(y_val, y_pred))

test_file = os.path.join(TRAIN_AUDIO_PATH, df.iloc[0]['fname'])
print("Prediction:", predict(test_file))
from IPython.display import Audio

file_path = os.path.join(TRAIN_AUDIO_PATH, df.iloc[0]['fname'])

print("Playing:", file_path)
Audio(file_path)

C:\Users\chaitu\AppData\Local\Temp\ipykernel_15632\599680939.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('label', group_keys=False).apply(


          fname         label  manually_verified
0  00044347.wav        Hi-hat                  0
1  001ca53d.wav     Saxophone                  1
2  002d256b.wav       Trumpet                  0
3  0033e230.wav  Glockenspiel                  1
4  00353774.wav         Cello                  1
label
Hi-hat                   300
Laughter                 300
Shatter                  300
Applause                 300
Squeak                   300
Acoustic_guitar          300
Bass_drum                300
Saxophone                300
Flute                    300
Double_bass              300
Tearing                  300
Fart                     300
Clarinet                 300
Fireworks                300
Trumpet                  300
Violin_or_fiddle         300
Cello                    300
Snare_drum               300
Oboe                     299
Gong                     292
Knock                    279
Writing                  270
Cough                    243
Bark                     239
Tamb

100%|██████████████████████████████████████████████████████████████████████████████| 1500/1500 [00:21<00:00, 68.23it/s]
C:\Users\chaitu\Documents\sample_project_1\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


X shape: (1500, 13, 32, 1)
y shape: (1500,)


Model: "sequential_22"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_51 (Conv2D)                   │ (None, 13, 32, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_51               │ (None, 13, 32, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_51 (MaxPooling2D)      │ (None, 6, 16, 32)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_52 (Conv2D)                   │ (None, 6, 16, 64)           │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_52               │ (None, 6, 16, 64)           │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_52 (MaxPooling2D)      │ (None, 3, 8, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_53 (Conv2D)                   │ (None, 3, 8, 128)           │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_53               │ (None, 3, 8, 128)           │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_53 (MaxPooling2D)      │ (None, 1, 4, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_17 (Flatten)                 │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_34 (Dropout)                 │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_49 (Dense)                     │ (None, 128)                 │          65,664 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_35 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_50 (Dense)                     │ (None, 5)                   │             645 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 159,877 (624.52 KB)

 Trainable params: 159,429 (622.77 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - accuracy: 0.3458 - loss: 1.8880 - val_accuracy: 0.4500 - val_loss: 1.3806
Epoch 2/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.4733 - loss: 1.3015 - val_accuracy: 0.5133 - val_loss: 1.1872
Epoch 3/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.5100 - loss: 1.1978 - val_accuracy: 0.5700 - val_loss: 1.1098
Epoch 4/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.5425 - loss: 1.1264 - val_accuracy: 0.5867 - val_loss: 1.1073
Epoch 5/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.5858 - loss: 1.0262 - val_accuracy: 0.6600 - val_loss: 0.9339
Epoch 6/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.6050 - loss: 0.9924 - val_accuracy: 0.6133 - val_loss: 1.0237
Epoch 7/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - accuracy: 0.6250 - loss: 0.9205 - val_accuracy: 0.6500 - val_loss: 0.9457
Epoch 8/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - accuracy: 0.6825 - loss: 0.8354 - val_accuracy: 0.6400 - v

In [ ]:
df.